In [108]:
# !pip install xgboost

In [109]:
import sys
print(sys.executable)

C:\Users\debas\AppData\Local\Programs\Python\Python311\python.exe


In [110]:
# C:\Users\debas\.conda\envs\multiflow_env\python.exe -m pip install xgboost

In [111]:
# C:\Users\debas\AppData\Local\Programs\Python\Python311\python.exe -m pip install mlflow

In [112]:
# import sys
# !{sys.executable} -m pip install mlflow   # The True Error less Code 

In [113]:
# import sys
# !{sys.executable} -m pip install dagshub

In [114]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression,Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.svm import SVR

from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from xgboost import XGBRegressor

import mlflow

In [115]:
# set the dagshub tracking server 
mlflow.set_tracking_uri("https://dagshub.com/Dev-debasish-09/uber-demand-prediction.mlflow")

In [116]:
import dagshub
dagshub.init(repo_owner='Dev-debasish-09', repo_name='uber-demand-prediction', mlflow=True)

Initialized MLflow to track repo "Dev-debasish-09/uber-demand-prediction"

Repository Dev-debasish-09/uber-demand-prediction initialized!

In [117]:
# load train and test data 
train_data_path = "C:\\Users\\debas\\Uber demand Prediction/Data/train.csv"
test_data_path = "C:\\Users\\debas\\Uber demand Prediction/Data/train.csv"

train_df = pd.read_csv(train_data_path,parse_dates=["tpep_pickup_datetime"]).set_index("tpep_pickup_datetime")
test_df = pd.read_csv(test_data_path,parse_dates=["tpep_pickup_datetime"]).set_index("tpep_pickup_datetime")

train_df

,lag_1,lag_2,lag_3,lag_4,region,total_pickups,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,,
2016-01-01 01:00:00,160.0,149.0,120.0,58.0,0,187,161.0,4
2016-01-01 01:15:00,187.0,160.0,149.0,120.0,0,194,175.0,4
2016-01-01 01:30:00,194.0,187.0,160.0,149.0,0,180,177.0,4
2016-01-01 01:45:00,180.0,194.0,187.0,160.0,0,197,185.0,4
2016-01-01 02:00:00,197.0,180.0,194.0,187.0,0,185,185.0,4
...,...,...,...,...,...,...,...,...
2016-02-29 22:45:00,15.0,9.0,11.0,11.0,29,12,12.0,0
2016-02-29 23:00:00,12.0,15.0,9.0,11.0,29,17,14.0,0
2016-02-29 23:15:00,17.0,12.0,15.0,9.0,29,15,14.0,0


In [118]:
# missing value in training data

train_df.isna().sum()

lag_1            0
lag_2            0
lag_3            0
lag_4            0
region           0
total_pickups    0
avg_pickups      0
day_of_week      0
dtype: int64

In [119]:
# missing value in testing data 

test_df.isna().sum()

lag_1            0
lag_2            0
lag_3            0
lag_4            0
region           0
total_pickups    0
avg_pickups      0
day_of_week      0
dtype: int64

In [120]:
# make x_train and y_train

x_train = train_df.drop(columns=["total_pickups"])

y_train = train_df["total_pickups"]

In [121]:
x_train.head()

,lag_1,lag_2,lag_3,lag_4,region,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,
2016-01-01 01:00:00,160.0,149.0,120.0,58.0,0,161.0,4
2016-01-01 01:15:00,187.0,160.0,149.0,120.0,0,175.0,4
2016-01-01 01:30:00,194.0,187.0,160.0,149.0,0,177.0,4
2016-01-01 01:45:00,180.0,194.0,187.0,160.0,0,185.0,4
2016-01-01 02:00:00,197.0,180.0,194.0,187.0,0,185.0,4


In [122]:
y_train

tpep_pickup_datetime
2016-01-01 01:00:00    187
2016-01-01 01:15:00    194
2016-01-01 01:30:00    180
2016-01-01 01:45:00    197
2016-01-01 02:00:00    185
                      ... 
2016-02-29 22:45:00     12
2016-02-29 23:00:00     17
2016-02-29 23:15:00     15
2016-02-29 23:30:00     15
2016-02-29 23:45:00     12
Name: total_pickups, Length: 172680, dtype: int64

In [123]:
# make X_test and y_test

x_test = test_df.drop(columns=["total_pickups"])

y_test = test_df["total_pickups"]

In [124]:
x_test.head()

,lag_1,lag_2,lag_3,lag_4,region,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,
2016-01-01 01:00:00,160.0,149.0,120.0,58.0,0,161.0,4
2016-01-01 01:15:00,187.0,160.0,149.0,120.0,0,175.0,4
2016-01-01 01:30:00,194.0,187.0,160.0,149.0,0,177.0,4
2016-01-01 01:45:00,180.0,194.0,187.0,160.0,0,185.0,4
2016-01-01 02:00:00,197.0,180.0,194.0,187.0,0,185.0,4


In [125]:
from sklearn import set_config

set_config(transform_output = "pandas")

In [126]:
# encode the data 

encoder = ColumnTransformer([
    ("ohe",OneHotEncoder(drop = "first",sparse_output = False),["region","day_of_week"])
],remainder = "passthrough",n_jobs = -1 ,force_int_remainder_cols = False)


In [127]:
encoder

ColumnTransformer(force_int_remainder_cols=False, n_jobs=-1,
                  remainder='passthrough',
                  transformers=[('ohe',
                                 OneHotEncoder(drop='first',
                                               sparse_output=False),
                                 ['region', 'day_of_week'])])

In [128]:
# encode the train and test data

x_train_encoded = encoder.fit_transform(x_train)
x_test_encoded = encoder.transform(x_test)

In [129]:
# !pip install optuna

In [130]:
import optuna
import tqdm

In [131]:
import mlflow
import os

# # Force MLflow to store runs locally inside your repo
# repo_path = r"C:/Users/debas/Dev-debasish-09/uber-demand-prediction/mlruns"
# mlflow.set_tracking_uri(f"file:///{repo_path.replace(os.sep, '/')}")

# # Now you can safely set your experiment
# mlflow.set_experiment("Model Selection")


In [132]:
# set the experiment

mlflow.set_experiment("Model Selection")

2025/09/01 15:30:37 INFO mlflow.tracking.fluent: Experiment with name 'Model Selection' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/cefc21b938fe45dd84d45fa8b9e501e6', creation_time=1756720745481, experiment_id='0', last_update_time=1756720745481, lifecycle_stage='active', name='Model Selection', tags={}>

In [133]:
def objective(trial):
    # start the child run
    with mlflow.start_run(nested=True) as child:
        
        # model name search space
        list_of_models = ["LR", "RF", "GBR", "XGBR"]
        model_name = trial.suggest_categorical("model_name", list_of_models)
    
        if model_name == "LR":
            model = LinearRegression()
    
        elif model_name == "RF":
            n_estimators_rf = trial.suggest_int("n_estimators_rf",10,100,step=10)
            max_depth_rf = trial.suggest_int("max_depth_rf",3,10)
            model = RandomForestRegressor(n_estimators=n_estimators_rf, 
                                          max_depth=max_depth_rf, 
                                          random_state=42, n_jobs=-1)
    
        elif model_name == "GBR":
            n_estimators_gb = trial.suggest_int("n_estimators_gb",10,100,step=10)
            learning_rate_gb = trial.suggest_float("learning_rate_gb",1e-4,1e-1, log=True)
            model = GradientBoostingRegressor(n_estimators=n_estimators_gb, 
                                              learning_rate=learning_rate_gb,
                                             random_state=42)
    
        elif model_name == "XGBR":
            n_estimators_xgb = trial.suggest_int("n_estimators_xgb",10,100,step=10)
            learning_rate_xgb = trial.suggest_float("learning_rate_xgb",1e-4,1e-1, log=True)
            max_depth_xgb = trial.suggest_int("max_depth_xgb",3,10)
            model = XGBRegressor(n_estimators=n_estimators_xgb,
                                learning_rate=learning_rate_xgb,
                                max_depth=max_depth_xgb)
    
        # log the model name
        mlflow.log_param("model_name",model_name)
        
        # log the model parameters
        mlflow.log_params(model.get_params())
        
        # fit on the data
        model.fit(x_train_encoded,y_train)
    
        # get the predictions
        y_pred = model.predict(x_test_encoded)
    
        # calculate the loss
        loss = mean_absolute_percentage_error(y_test, y_pred)
    
        # log the metric
        mlflow.log_metric("MAPE",loss)
        return loss

In [134]:
# optimize the objective function

with mlflow.start_run(run_name="best_model", nested=True) as parent:

    # create a study object
    study = optuna.create_study(study_name="model_selection", direction="minimize")
    # optimize the objective function
    study.optimize(func=objective, n_trials=100, n_jobs=-1)
    
    # log the best parameters
    mlflow.log_params(study.best_params)
    # log the best error value
    mlflow.log_metric("Best_MAPE", study.best_value)

[I 2025-09-01 15:30:38,929] A new study created in memory with name: model_selection


🏃 View run industrious-seal-130 at: https://dagshub.com/Dev-debasish-09/uber-demand-prediction.mlflow/#/experiments/0/runs/577ab7376abe4bfca3c5e662d7ef6a27
🧪 View experiment at: https://dagshub.com/Dev-debasish-09/uber-demand-prediction.mlflow/#/experiments/0


[I 2025-09-01 15:31:09,653] Trial 1 finished with value: 0.08778013304566522 and parameters: {'model_name': 'LR'}. Best is trial 1 with value: 0.08778013304566522.


🏃 View run nosy-quail-888 at: https://dagshub.com/Dev-debasish-09/uber-demand-prediction.mlflow/#/experiments/0/runs/e52973bd85a04693a88f53da1849733c
🧪 View experiment at: https://dagshub.com/Dev-debasish-09/uber-demand-prediction.mlflow/#/experiments/0
🏃 View run efficient-finch-772 at: https://dagshub.com/Dev-debasish-09/uber-demand-prediction.mlflow/#/experiments/0/runs/606b2d073dea4610af7f118c2af6487c
🧪 View experiment at: https://dagshub.com/Dev-debasish-09/uber-demand-prediction.mlflow/#/experiments/0
🏃 View run selective-crow-985 at: https://dagshub.com/Dev-debasish-09/uber-demand-prediction.mlflow/#/experiments/0/runs/91e7a9e652854f7fb3f37b5ecca819ef
🧪 View experiment at: https://dagshub.com/Dev-debasish-09/uber-demand-prediction.mlflow/#/experiments/0
🏃 View run selective-hen-817 at: https://dagshub.com/Dev-debasish-09/uber-demand-prediction.mlflow/#/experiments/0/runs/339d4de9128d4bf18f0a39d4775036d5
🧪 View experiment at: https://dagshub.com/Dev-debasish-09/uber-demand-predic

[I 2025-09-01 15:31:55,629] Trial 0 finished with value: 0.08778013304566522 and parameters: {'model_name': 'LR'}. Best is trial 1 with value: 0.08778013304566522.
[I 2025-09-01 15:31:56,667] Trial 7 finished with value: 0.12394048466555928 and parameters: {'model_name': 'RF', 'n_estimators_rf': 90, 'max_depth_rf': 9}. Best is trial 1 with value: 0.08778013304566522.
[I 2025-09-01 15:31:57,676] Trial 12 finished with value: 0.08778013304566522 and parameters: {'model_name': 'LR'}. Best is trial 1 with value: 0.08778013304566522.


...

🏃 View run kindly-dove-258 at: https://dagshub.com/Dev-debasish-09/uber-demand-prediction.mlflow/#/experiments/0/runs/59607f81743d49a7b297452c5c17cd46
🧪 View experiment at: https://dagshub.com/Dev-debasish-09/uber-demand-prediction.mlflow/#/experiments/0
🏃 View run clumsy-carp-854 at: https://dagshub.com/Dev-debasish-09/uber-demand-prediction.mlflow/#/experiments/0/runs/9c0b8ee1a0a14e2d8ee04833749bc34a
🧪 View experiment at: https://dagshub.com/Dev-debasish-09/uber-demand-prediction.mlflow/#/experiments/0


[I 2025-09-01 15:43:43,610] Trial 98 finished with value: 0.08778013304566522 and parameters: {'model_name': 'LR'}. Best is trial 1 with value: 0.08778013304566522.
[I 2025-09-01 15:43:44,640] Trial 99 finished with value: 0.08778013304566522 and parameters: {'model_name': 'LR'}. Best is trial 1 with value: 0.08778013304566522.


🏃 View run valuable-ape-410 at: https://dagshub.com/Dev-debasish-09/uber-demand-prediction.mlflow/#/experiments/0/runs/9b16f46775284c2db765be4b59c26723
🧪 View experiment at: https://dagshub.com/Dev-debasish-09/uber-demand-prediction.mlflow/#/experiments/0


[I 2025-09-01 15:44:14,128] Trial 76 finished with value: 0.08778013304566522 and parameters: {'model_name': 'LR'}. Best is trial 1 with value: 0.08778013304566522.


🏃 View run best_model at: https://dagshub.com/Dev-debasish-09/uber-demand-prediction.mlflow/#/experiments/0/runs/1f4fe927c1df441b8c5e113735f35665
🧪 View experiment at: https://dagshub.com/Dev-debasish-09/uber-demand-prediction.mlflow/#/experiments/0


In [135]:
study.best_value

0.08778013304566522

In [136]:
# best parameters

study.best_params

{'model_name': 'LR'}

In [137]:
# model value counts

study.trials_dataframe()['params_model_name'].value_counts()

params_model_name
LR      63
XGBR    14
RF      12
GBR     11
Name: count, dtype: int64

In [138]:
from optuna.visualization import (
    plot_optimization_history, 
    plot_parallel_coordinate, 
    plot_param_importances
)

In [139]:
plot_optimization_history(study)

In [140]:
plot_parallel_coordinate(study, params=["model_name"])

In [141]:
# train the linear regression model

lr = LinearRegression()

lr.fit(x_train_encoded, y_train)

# get predictions
y_pred_train = lr.predict(x_train_encoded) 
y_pred_test = lr.predict(x_test_encoded)

# loss

mape_train = mean_absolute_percentage_error(y_train, y_pred_train)
mape_test = mean_absolute_percentage_error(y_test, y_pred_test)

print("The training error is ", mape_train)
print("The test error is ", mape_test)

The training error is  0.08778013304566522
The test error is  0.08778013304566522


In [142]:
lr.coef_

array([-2.33737604,  0.71512405, -0.55601505, -1.25311068, -3.20463231,
       -0.86685973, -2.79925402, -3.62516859,  0.41386463, -2.9376376 ,
       -1.97624678, -3.75050442,  0.51806283, -2.54033388, -2.43297463,
        0.47632075,  0.61254786, -4.7417372 , -2.03077217, -1.26960984,
       -4.03690273, -2.08863167, -1.0414428 ,  0.73561736, -0.99999442,
       -0.85944985, -2.43098478,  0.67112238,  0.57385071, -0.11719951,
       -0.28045898, -0.37180749, -0.5238324 , -0.4233113 , -0.34045774,
       -0.54170892, -0.36264553, -0.2493965 , -0.31905518,  2.4912456 ])

In [143]:
def tune_ridge(trial):
    # hyperparameter space
    alpha = trial.suggest_float("alpha",30,100)
    
    # make the model object
    ridge = Ridge(alpha=alpha, random_state=42)
    
    # train the model
    ridge.fit(x_train_encoded, y_train)
    
    # get predictions
    y_pred = ridge.predict(x_test_encoded)
    
    # calculate loss
    loss = mean_absolute_percentage_error(y_test, y_pred)

    return loss
        

In [144]:
# create study

study = optuna.create_study(study_name="tune_model", direction="minimize")

[I 2025-09-01 15:44:17,758] A new study created in memory with name: tune_model


In [145]:
# optimize

study.optimize(func=tune_ridge, n_trials=100, n_jobs=-1, show_progress_bar=True)

  0%|          | 0/100 [00:00<?, ?it/s]

[I 2025-09-01 15:44:18,204] Trial 0 finished with value: 0.08747009344521514 and parameters: {'alpha': 74.50896540693373}. Best is trial 0 with value: 0.08747009344521514.
[I 2025-09-01 15:44:18,341] Trial 1 finished with value: 0.08739488872277677 and parameters: {'alpha': 99.42091167119598}. Best is trial 1 with value: 0.08739488872277677.
[I 2025-09-01 15:44:18,343] Trial 2 finished with value: 0.08749079157710062 and parameters: {'alpha': 68.19680631119674}. Best is trial 1 with value: 0.08739488872277677.
[I 2025-09-01 15:44:18,375] Trial 3 finished with value: 0.08745997531494708 and parameters: {'alpha': 77.69279321173718}. Best is trial 1 with value: 0.08739488872277677.
[I 2025-09-01 15:44:18,415] Trial 9 finished with value: 0.08742302896777865 and parameters: {'alpha': 89.75903193352147}. Best is trial 1 with value: 0.08739488872277677.
[I 2025-09-01 15:44:18,425] Trial 7 finished with value: 0.08749318064757068 and parameters: {'alpha': 67.48263889397481}. Best is trial 1 w

In [146]:
# best parameters

study.best_params

{'alpha': 99.99489988072378}

In [147]:
# best value

study.best_value

0.08739325736771568